# Volume-alignment experiment — paragraph-level evaluation (EN & PL)
Evaluates EN(145) and PL(145) down-aligned models using the same pipeline as the main eval notebook.

**Pipeline identical to the main eval notebook:**
1. 从S3下载checkpoint
2. 加载dev文章 (从SemEval原始目录)
3. 基于概率分布优化阈值
4. 生成段落级预测
5. 对比gold标准计算 Macro/Micro F1


In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory: contains data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# Training data directories
TRAIN_ARTICLES_DIR = "your/path/here"  # multilingual training articles
TRAIN_LABELS_FILE  = "your/path/here"  # corresponding label file
DEV_ARTICLES_DIR   = "your/path/here"  # dev articles
DEV_LABELS_FILE    = "your/path/here"  # dev labels

# SemEval-2023 dev data (per-language evaluation)
SEMEVAL_DEV_DIR = "your/path/here"  # contains en/, po/, ru/ subdirs

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


## 1. 导入库和环境配置

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import boto3
import tempfile
import traceback
from transformers import AutoModel, AutoModelForSequenceClassification, AutoConfig, AutoTokenizer
import pytorch_lightning as pl

# Environment variables
os.environ["TOKENIZERS_PARALLELISM"] = 'false'
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. 配置参数

**Only differences from the main eval notebook: model filename, language code, gold file path.**

In [ ]:
import os, torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model hyperparameters (must match training)
model_name = 'xlm-roberta-base'
MAX_LENGTH = 256
MAX_EXPLANATION_LENGTH = 128

# S3 configuration (credentials via env vars)
ENDPOINT  = "https://your-s3-endpoint"
S3_BUCKET = "your-s3-bucket"
S3_PREFIX = "multi_label_models/downward_align"
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

SEEDS = [42, 123, 456]   # 3 seeds matching the training runs

# 工具文件
TECHNIQUES_FILE   = "your/path/here"
EXPLANATIONS_FILE = "your/path/here"

# Gold 标准文件
EN_GOLD_PATH = "your/path/here"
PO_GOLD_PATH = "your/path/here"

# dev文章目录
TEST_FOLDERS = {
    'en': "your/path/here",
    'po': "your/path/here",
}

BATCH_SIZE = 16
THRESHOLD  = 0.5

OUTPUT_DIR = "your/path/here"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"SEEDS  : {SEEDS}")
print(f"Output : {OUTPUT_DIR}")
print("Config done")

## 3. 模型定义（与训练notebook完全一致）

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 模型定义
# ═══════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """Focal Loss实现"""
    def __init__(self, alpha=1, gamma=2, pos_weight=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        epsilon = 1e-7
        targets_stable = torch.clamp(targets.float(), min=epsilon, max=1-epsilon)
        
        if self.pos_weight is not None:
            BCE_loss = F.binary_cross_entropy_with_logits(
                inputs, targets_stable, pos_weight=self.pos_weight, reduction='none'
            )
        else:
            BCE_loss = F.binary_cross_entropy_with_logits(
                inputs, targets_stable, reduction='none'
            )
        
        pt = torch.exp(-BCE_loss)
        F_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss


class MultiLabelClassifierWithExplanations(pl.LightningModule):
    """双编码器多标签分类模型"""
    
    def __init__(self, plm, num_labels, class_weights=None, learning_rate=1e-5, 
                 warmup_steps=1000, loss_type='bce', focal_gamma=2.0, focal_alpha=1.0):
        super().__init__()
        
        # 双编码器
        self.text_encoder = AutoModel.from_pretrained(plm)
        self.explanation_encoder = AutoModel.from_pretrained(plm)
        
        # 启用梯度检查点
        if hasattr(self.text_encoder, "gradient_checkpointing_enable"):
            self.text_encoder.gradient_checkpointing_enable()
            self.explanation_encoder.gradient_checkpointing_enable()
        
        # 模型参数
        self.hidden_size = self.text_encoder.config.hidden_size
        self.num_labels = num_labels
        self.learning_rate = learning_rate
        self.warmup_steps = warmup_steps
        self.loss_type = loss_type
        
        # 交叉注意力层
        self.text_to_exp_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size,
            num_heads=8,
            batch_first=True
        )
        
        self.exp_to_text_attention = nn.MultiheadAttention(
            embed_dim=self.hidden_size,
            num_heads=8,
            batch_first=True
        )
        
        # 特征融合层
        self.fusion_layer = nn.Sequential(
            nn.Linear(self.hidden_size * 4, self.hidden_size * 2),
            nn.LayerNorm(self.hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        
        # 分类层
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        
        # 损失函数
        if loss_type == 'focal':
            self.criterion = FocalLoss(
                alpha=focal_alpha, 
                gamma=focal_gamma, 
                pos_weight=class_weights
            )
        else:
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    
    def forward(self, text_ids, text_mask, exp_ids=None, exp_mask=None):
        # 文本编码
        text_outputs = self.text_encoder(
            input_ids=text_ids, 
            attention_mask=text_mask,
            return_dict=True
        )
        text_hidden = text_outputs.last_hidden_state
        text_cls = text_hidden[:, 0, :]
        
        # 如果没有解释，直接使用文本
        if exp_ids is None or exp_mask is None:
            logits = self.classifier(text_cls)
            return logits
        
        # 解释编码
        exp_outputs = self.explanation_encoder(
            input_ids=exp_ids, 
            attention_mask=exp_mask,
            return_dict=True
        )
        exp_hidden = exp_outputs.last_hidden_state
        exp_cls = exp_hidden[:, 0, :]
        
        # 交叉注意力
        text_to_exp_out, _ = self.text_to_exp_attention(
            query=text_hidden,
            key=exp_hidden,
            value=exp_hidden,
            key_padding_mask=~exp_mask.bool()
        )
        
        exp_to_text_out, _ = self.exp_to_text_attention(
            query=exp_hidden,
            key=text_hidden,
            value=text_hidden,
            key_padding_mask=~text_mask.bool()
        )
        
        # 提取CLS特征
        text_to_exp_cls = text_to_exp_out[:, 0, :]
        exp_to_text_cls = exp_to_text_out[:, 0, :]
        
        # 特征融合
        combined_feature = torch.cat([
            text_cls,
            exp_cls,
            text_to_exp_cls,
            exp_to_text_cls
        ], dim=1)
        
        fused_feature = self.fusion_layer(combined_feature)
        logits = self.classifier(fused_feature)
        
        return logits
    
    def configure_optimizers(self):
        # 测试时不需要，但加载checkpoint需要这个方法
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        return optimizer


print("✓ 模型类定义完成")

## 4. 辅助函数

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 添加新的辅助函数
# ═══════════════════════════════════════════════════════════════

def compute_class_weights(true_labels, neg_pos_ratio=3.0, max_weight=20.0):
    """计算类别权重，处理类别不平衡"""
    print("计算类别权重...")
    
    # 统计每个标签的正样本数量
    pos_counts = np.sum(true_labels, axis=0)
    total_samples = len(true_labels)
    neg_counts = total_samples - pos_counts
    
    # 计算权重
    weights = []
    for i in range(true_labels.shape[1]):
        if pos_counts[i] > 0:
            weight = min((neg_counts[i] / pos_counts[i]) * neg_pos_ratio, max_weight)
            weights.append(weight)
        else:
            weights.append(1.0)
    
    print(f"类别权重范围: {min(weights):.2f} - {max(weights):.2f}")
    return torch.tensor(weights, dtype=torch.float32)


def optimize_thresholds(model, tokenizer, test_df, labels_name, explanations=None):
    """
    为每个标签优化阈值
    注意：这里使用测试集的预测概率分布来优化阈值（无监督方式）
    """
    print("\n" + "="*70)
    print("优化标签阈值（基于预测概率分布）")
    print("="*70)
    
    model.eval()
    all_probabilities = []
    
    # 第一遍：获取所有样本的预测概率
    print("正在收集预测概率...")
    for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="获取概率"):
        batch = test_df.iloc[i:i+BATCH_SIZE]
        batch_texts = batch['text'].tolist()
        
        # 准备解释数据
        batch_exps = []
        for _, row in batch.iterrows():
            key = (row['id'], row['text'])
            exp = explanations.get(key, "") if explanations else ""
            batch_exps.append(exp if exp else "")
        
        try:
            # 文本编码
            text_inputs = tokenizer(
                batch_texts,
                padding="max_length",
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            ).to(device)
            
            # 解释编码
            exp_inputs = tokenizer(
                batch_exps,
                padding="max_length",
                truncation=True,
                max_length=MAX_EXPLANATION_LENGTH,
                return_tensors="pt"
            ).to(device)
            
            # 预测
            with torch.no_grad():
                logits = model(
                    text_ids=text_inputs["input_ids"],
                    text_mask=text_inputs["attention_mask"],
                    exp_ids=exp_inputs["input_ids"],
                    exp_mask=exp_inputs["attention_mask"]
                )
                probabilities = torch.sigmoid(logits)
                all_probabilities.append(probabilities.cpu().numpy())
            
            # 清理内存
            if i % 50 == 0:
                torch.cuda.empty_cache()
                
        except Exception as e:
            print(f"批次 {i//BATCH_SIZE} 处理出错: {str(e)}")
            empty_probs = np.zeros((len(batch), len(labels_name)))
            all_probabilities.append(empty_probs)
    
    # 合并所有概率
    all_probabilities = np.concatenate(all_probabilities, axis=0)
    
    # 为每个标签基于概率分布优化阈值
    optimized_thresholds = {}
    
    print("\n为每个标签选择阈值:")
    for i, label in enumerate(labels_name):
        label_probs = all_probabilities[:, i]
        
        # 分析概率分布
        prob_mean = label_probs.mean()
        prob_median = np.median(label_probs)
        prob_75 = np.percentile(label_probs, 75)
        prob_90 = np.percentile(label_probs, 90)
        
        # 策略：使用75分位数和90分位数之间的值
        # 如果大部分概率都很低，使用较低的阈值
        if prob_90 < 0.3:
            threshold = max(0.25, prob_75)
        elif prob_90 < 0.5:
            threshold = max(0.3, prob_75)
        else:
            threshold = max(0.35, min(0.55, (prob_75 + prob_90) / 2))
        
        optimized_thresholds[label] = threshold
        
        # 计算使用这个阈值会预测多少样本
        predicted_count = (label_probs > threshold).sum()
        
        print(f"  {label:35s}: "
              f"阈值={threshold:.2f} | "
              f"均值={prob_mean:.3f} | "
              f"中位数={prob_median:.3f} | "
              f"75%={prob_75:.3f} | "
              f"预测数={predicted_count}")
    
    return optimized_thresholds, all_probabilities

In [ ]:
import boto3, tempfile, traceback

def download_from_s3(bucket_name, s3_key, local_file_path):
    """从S3下载模型"""
    try:
        s3 = boto3.resource('s3', endpoint_url=ENDPOINT,
            aws_access_key_id=AWS_ACCESS_KEY_ID,
            aws_secret_access_key=AWS_SECRET_ACCESS_KEY)
        os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
        print(f"正在从S3下载模型...")
        s3.Bucket(bucket_name).download_file(s3_key, local_file_path)
        print(f"✓ 模型下载成功")
        return True
    except Exception as e:
        print(f"✗ 下载失败: {str(e)}")
        return False


def load_label_classes():
    """加载标签类别"""
    labels_name = []
    with open(TECHNIQUES_FILE, "r", encoding='utf-8') as f:
        for line in f.readlines():
            labels_name.append(line.rstrip())
    labels_name.sort()
    print(f"✓ 加载了 {len(labels_name)} 个标签")
    return labels_name


def load_test_data(test_folder):
    """加载指定语言的dev文章 — 与en677_eval完全一致"""
    print(f"\n正在从以下路径加载测试数据:")
    print(f"  {test_folder}")

    text_data = []
    files = [f for f in os.listdir(test_folder)
             if f.endswith('.txt') and not f.startswith('._')]
    print(f"✓ 找到 {len(files)} 个文本文件")

    for fil in tqdm(files, desc="读取文件"):
        article_id = fil.replace('.txt', '')
        try:
            with open(os.path.join(test_folder, fil), 'r', encoding='utf-8') as f:
                content = f.read()
            for line_num, line_text in enumerate(content.splitlines(), 1):
                line_text = line_text.strip()
                if line_text:
                    text_data.append({'id': article_id, 'line': line_num, 'text': line_text})
        except Exception as e:
            print(f"⚠ 读取文件 {fil} 失败: {str(e)}")

    df = pd.DataFrame(text_data)
    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    original_len = len(df)
    df = df[df['text'].str.len() <= 1000].copy()
    if len(df) < original_len:
        print(f"⚠ 过滤了 {original_len - len(df)} 条超长文本")
    print(f"✓ 加载了 {len(df)} 条测试样本")
    return df


def load_explanations():
    """加载解释数据"""
    explanations = {}
    if not os.path.exists(EXPLANATIONS_FILE):
        print("⚠ 解释文件不存在，使用空解释")
        return explanations
    try:
        df = pd.read_csv(EXPLANATIONS_FILE, sep='\t')
        for _, row in df.iterrows():
            explanations[(row['id'], row['text'])] = row['analysis']
        print(f"✓ 加载了 {len(explanations)} 条解释数据")
    except Exception as e:
        print(f"⚠ 加载解释数据失败: {str(e)}")
    return explanations


def load_gold_labels(gold_path):
    """加载段落级gold标准: article_id \t line_num [\t tech1,tech2,...]"""
    para_labels = {}   # {(article_id, line_num): set(techniques)}
    articles    = set()
    with open(gold_path) as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) < 2: continue
            raw_id = parts[0]
            lnum   = int(parts[1])
            techs  = set(parts[2].split(',')) - {''} if len(parts) >= 3 else set()
            para_labels[(raw_id, lnum)] = techs
            articles.add(raw_id)
    print(f"✓ Gold: {len(articles)} 篇文章, {len(para_labels)} 段落, "
          f"{sum(1 for t in para_labels.values() if t)} 含技术段落")
    return para_labels, articles

print("✓ 辅助函数定义完成")


## 主测试流程

In [ ]:
print("\n" + "="*70)
print("第1步：加载标签和explanations（模型在评估循环里按seed加载）")
print("="*70)

labels_name  = load_label_classes()
num_labels   = len(labels_name)
tokenizer    = AutoTokenizer.from_pretrained(model_name)
explanations = load_explanations()

In [ ]:
def evaluate_language(lang_code, test_folder, gold_path, seed=None):
    """
    对指定语言完整执行en677_eval的全部步骤，返回Macro/Micro F1。
    步骤与en677_eval完全一致：加载数据→优化阈值→生成预测→计算F1
    """
    print("\n" + "="*70)
    print(f"评估语言: {lang_code.upper()}  |  seed={seed}  |  {test_folder}")
    print("="*70)

    # ── 第2步：加载测试数据 ──────────────────────────────────────
    print("\n第2步：加载测试数据")
    test_df = load_test_data(test_folder)

    # ── article_id标准化：去掉可能的'article'前缀，统一为纯数字 ──
    # SemEval文件名可能是 'article813452859.txt' → id='article813452859'
    # Gold文件里是纯数字 '813452859'
    # 统一处理：去掉'article'前缀
    def normalize_id(aid):
        s = str(aid)
        return s[7:] if s.startswith('article') else s

    test_df['id_norm'] = test_df['id'].apply(normalize_id)

    # 打印一下实际值，方便确认
    sample_ids = test_df['id'].unique()[:3].tolist()
    sample_norm = test_df['id_norm'].unique()[:3].tolist()
    print(f"  test_df id 示例: {sample_ids}")
    print(f"  标准化后:         {sample_norm}")

    # ── 第3步：优化阈值（与en677_eval完全一致）────────────────────
    print("\n第3步：优化阈值（基于预测概率分布）")
    optimized_thresholds, all_probabilities = optimize_thresholds(
        model, tokenizer, test_df, labels_name, explanations)
    print(f"阈值范围: {min(optimized_thresholds.values()):.2f} - "
          f"{max(optimized_thresholds.values()):.2f}")

    # ── 第4步：使用优化阈值生成预测 ──────────────────────────────
    print("\n第4步：使用优化阈值生成预测")
    predictions = np.zeros_like(all_probabilities)
    for i, label in enumerate(labels_name):
        predictions[:, i] = (all_probabilities[:, i] > optimized_thresholds[label]).astype(int)
    print(f"✓ 预测完成  平均标签数/段落: {predictions.sum(axis=1).mean():.2f}")

    # ── 第5步：对比Gold标准计算F1 ─────────────────────────────────
    print("\n第5步：对比Gold标准计算F1")
    gold_labels, _ = load_gold_labels(gold_path)

    # gold里的article_id也标准化（去掉'article'前缀，保留纯数字）
    gold_norm = {}
    for (raw_id, lnum), techs in gold_labels.items():
        gold_norm[(normalize_id(raw_id), lnum)] = techs

    all_techs = sorted({t for ts in gold_norm.values() for t in ts})
    print(f"  Gold技术类型数: {len(all_techs)}")

    # 用标准化id在test_df中查找
    # 建立 (id_norm, line) → 行索引 的快速查找表
    lookup = {(row['id_norm'], row['line']): i
              for i, row in test_df.reset_index(drop=True).iterrows()}

    y_true, y_pred = [], []
    missing_in_pred = 0
    matched = 0

    for (norm_id, lnum), gold_techs in gold_norm.items():
        true_vec = [1 if t in gold_techs else 0 for t in all_techs]

        if (norm_id, lnum) in lookup:
            idx = lookup[(norm_id, lnum)]
            pred_techs = {labels_name[k] for k, p in enumerate(predictions[idx]) if p == 1}
            pred_vec   = [1 if t in pred_techs else 0 for t in all_techs]
            matched += 1
        else:
            pred_vec = [0] * len(all_techs)
            missing_in_pred += 1

        y_true.append(true_vec)
        y_pred.append(pred_vec)

    print(f"  匹配段落: {matched} | 未匹配(预测为空): {missing_in_pred}")

    if matched == 0:
        print("  ❌ 匹配数为0！检查article_id和line_num格式")
        print(f"  Gold示例: {list(gold_norm.keys())[:3]}")
        print(f"  test_df示例: {list(lookup.keys())[:3]}")
        return {'macro_f1': 0, 'micro_f1': 0, 'weighted_f1': 0, 'binary_f1': 0, 'tech_f1': {}}

    macro_f1    = f1_score(y_true, y_pred, average='macro',    zero_division=0)
    micro_f1    = f1_score(y_true, y_pred, average='micro',    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    bin_true  = [1 if any(v) else 0 for v in y_true]
    bin_pred  = [1 if any(v) else 0 for v in y_pred]
    binary_f1 = f1_score(bin_true, bin_pred, zero_division=0)

    print(f"\n{'='*55}")
    print(f"  {lang_code.upper()} 评估结果")
    print(f"{'='*55}")
    print(f"  Macro  F1 : {macro_f1*100:.2f}%")
    print(f"  Micro  F1 : {micro_f1*100:.2f}%")
    print(f"  Weighted F1: {weighted_f1*100:.2f}%")
    print(f"  Binary F1 : {binary_f1*100:.2f}%")

    per_tech_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    tech_rows   = sorted(zip(all_techs, per_tech_f1), key=lambda x: -x[1])
    print(f"\n  Per-technique F1:")
    for tech, f1 in tech_rows:
        bar = '█' * int(f1 * 20)
        print(f"    {tech:<45} {f1:.3f}  {bar}")

    threshold_file = os.path.join(OUTPUT_DIR, f'downalign_{lang_code}_optimized_thresholds.json')
    import json as _json
    with open(threshold_file, 'w') as fp:
        _json.dump(optimized_thresholds, fp, indent=2, ensure_ascii=False)
    print(f"\n✓ 阈值已保存: {threshold_file}")

    return {
        'macro_f1': macro_f1, 'micro_f1': micro_f1,
        'weighted_f1': weighted_f1, 'binary_f1': binary_f1,
        'tech_f1': dict(zip(all_techs, per_tech_f1.tolist()))
    }

print("✓ evaluate_language() 定义完成")

## 执行评估

In [ ]:
# ── 3个seed分别评估EN ──────────────────────────────────────────
en_results_all = []

for seed in SEEDS:
    ckpt_s3_key   = f"{S3_PREFIX}/xlm-roberta-base_10_bce_downalign_seed{seed}.ckpt"
    ckpt_local    = f"{OUTPUT_DIR}/downalign_seed{seed}.ckpt"

    # 下载checkpoint（已存在则跳过）
    if not os.path.exists(ckpt_local):
        download_from_s3(S3_BUCKET, ckpt_s3_key, ckpt_local)

    # 加载模型
    model = MultiLabelClassifierWithExplanations.load_from_checkpoint(
        ckpt_local, plm=model_name, num_labels=num_labels, strict=False)
    model.eval().to(device)
    print(f"\n✓ seed={seed} 模型加载完成")

    res = evaluate_language('en', TEST_FOLDERS['en'], EN_GOLD_PATH, seed=seed)
    en_results_all.append(res)

import numpy as np
print("\n=== EN 3-seed 汇总 ===")
for key in ['macro_f1', 'micro_f1', 'binary_f1']:
    vals = [r[key] for r in en_results_all]
    print(f"  {key}: {np.mean(vals)*100:.2f}% ± {np.std(vals)*100:.2f}%")

In [ ]:
# ── 3个seed分别评估PO ──────────────────────────────────────────
po_results_all = []

for seed in SEEDS:
    ckpt_local = f"{OUTPUT_DIR}/downalign_seed{seed}.ckpt"
    # checkpoint已在EN循环里下载，直接加载
    model = MultiLabelClassifierWithExplanations.load_from_checkpoint(
        ckpt_local, plm=model_name, num_labels=num_labels, strict=False)
    model.eval().to(device)
    print(f"\n✓ seed={seed} 模型加载完成")

    res = evaluate_language('po', TEST_FOLDERS['po'], PO_GOLD_PATH, seed=seed)
    po_results_all.append(res)

print("\n=== PO 3-seed 汇总 ===")
for key in ['macro_f1', 'micro_f1', 'binary_f1']:
    vals = [r[key] for r in po_results_all]
    print(f"  {key}: {np.mean(vals)*100:.2f}% ± {np.std(vals)*100:.2f}%")

## 最终对比表

In [ ]:
import numpy as np, json as _json

def mean_std(results_all, key):
    vals = [r[key] for r in results_all]
    return np.mean(vals), np.std(vals)

en_macro,  en_macro_std  = mean_std(en_results_all, 'macro_f1')
en_micro,  en_micro_std  = mean_std(en_results_all, 'micro_f1')
en_binary, en_binary_std = mean_std(en_results_all, 'binary_f1')
po_macro,  po_macro_std  = mean_std(po_results_all, 'macro_f1')
po_micro,  po_micro_std  = mean_std(po_results_all, 'micro_f1')
po_binary, po_binary_std = mean_std(po_results_all, 'binary_f1')

print("\n" + "="*75)
print("  volume-alignment (down-aligned)实验 vs 论文Table 1 (Sup-FT, mean±std over 3 seeds)")
print("="*75)
print(f"  {'设置':<28} {'Macro F1':>16} {'Micro F1':>16} {'Binary F1':>16}")
print(f"  {'-'*76}")
print(f"  {'EN(446) full  [paper]':<28} {'41.76%':>16} {'59.15%':>16} {'97.11%':>16}")
print(f"  {'EN(145) down-aligned':<28} "
      f"{en_macro*100:>8.2f}±{en_macro_std*100:.2f}% "
      f"{en_micro*100:>8.2f}±{en_micro_std*100:.2f}% "
      f"{en_binary*100:>8.2f}±{en_binary_std*100:.2f}%")
print(f"  {'-'*76}")
print(f"  {'PL(674) full  [paper]':<28} {'52.23%':>16} {'66.75%':>16} {'96.77%':>16}")
print(f"  {'PL(145) down-aligned':<28} "
      f"{po_macro*100:>8.2f}±{po_macro_std*100:.2f}% "
      f"{po_micro*100:>8.2f}±{po_micro_std*100:.2f}% "
      f"{po_binary*100:>8.2f}±{po_binary_std*100:.2f}%")
print("="*75)

gap = po_macro - en_macro
print(f"\n解读:")
print(f"  EN(446)→EN(145) Macro F1 下降: {(0.4176-en_macro)*100:.2f}pp")
print(f"  EN(145) vs PL(145) gap:        {gap*100:+.2f}pp ({'PL更强' if gap>0 else 'EN更强'})")
if gap > 0.03:
    print("  → 即使数据量相同PL仍显著更强，强化resource curse假说")
elif gap < -0.03:
    print("  → EN在145篇时反超PL，resource curse假说被削弱")
else:
    print("  → EN/PL表现相近，数据量是主要confound")

out = {
    'en_down145': {'macro': f"{en_macro:.4f}±{en_macro_std:.4f}",
                   'micro': f"{en_micro:.4f}±{en_micro_std:.4f}",
                   'binary': f"{en_binary:.4f}±{en_binary_std:.4f}",
                   'per_seed': en_results_all},
    'po_down145': {'macro': f"{po_macro:.4f}±{po_macro_std:.4f}",
                   'micro': f"{po_micro:.4f}±{po_micro_std:.4f}",
                   'binary': f"{po_binary:.4f}±{po_binary_std:.4f}",
                   'per_seed': po_results_all},
}
with open(os.path.join(OUTPUT_DIR, 'downward_align_eval_results_3seeds.json'), 'w') as fp:
    _json.dump(out, fp, indent=2, ensure_ascii=False, default=float)
print("\n✓ 结果已保存:", os.path.join(OUTPUT_DIR, 'downward_align_eval_results_3seeds.json'))
# --- Expected results (实验结果汇总 §down-align) ---
# EN(145) Macro F1 = 10.91% ± 1.11pp  (mean over seeds 42/123/456)
# PL(145) Macro F1 = 15.64% ± 1.36pp
# PL-EN gap = +4.72pp,  paired t-test: t=8.83, p=0.013
